# E1.8 · Third-party and model supply chain risk

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.7 · Continuous control verification](https://spbreed.github.io/cyber-commons/lessons/E1.7.html)**.

| | |
|---|---|
| Tools used | OWASP AIBOM, Sigstore |

## What this lesson is

**What it covers.** Run a real AIBOM against a vendor model artefact.

**Why a security engineer needs it.** Vendor AI features enabled by default; sub-processor chains you never mapped. The control it builds is: questions that actually discriminate between vendors.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Your model vendor, your hosting, your adapters and your MCP servers are all somebody else's risk decisions, inherited. Diligence questions that produce real answers are specific; the generic questionnaire produces a filing.

> **At CyberTravels.** CyberTravels inherited its model vendor's decisions, its OCR library's, and a third-party MCP server's. R4.

## 2 · The framework

```
   inherited decisions

   model vendor  --> training data, safety posture, retention
   hosting       --> where inference happens, what is logged
   adapters      --> who built them, against which base
   agent tooling --> MCP servers, their tool descriptions

   generic questionnaire -> a filing
   specific question     -> an answer you can act on
```

Third-party risk for AI has the ordinary supply-chain problem plus a question
nobody's assessment form asks:

> **Can this component change without telling us?**

For a library the answer is no — you pin a version. For a hosted model the
answer is usually yes, and it changes the risk rating, because every control you
tested was tested against behaviour the vendor can replace on a Tuesday.

Three artefact classes, with genuinely different maturity:

- **Libraries** — signing, version pinning, download signals. Mature.
- **Model weights or a hosted model** — attestation possible and rare; no
  popularity signal that means anything; version stability is a contractual
  question, not a technical one.
- **Prompt and tool packages (MCP, skills)** — no signing convention, and they
  run with your agent's authority.

Saying which signals are unavailable is part of the assessment, not a gap in it.

## 3 · The procedure, as a skill

The skill scores each AI component on the two properties that make it different — silent change, and running with the agent's authority — then invalidates every control test taken before the model changed, because a test against a different model is evidence about something else.

In [ ]:
# skills/grc/third-party-ai-assessment/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: third-party-ai-assessment
description: >-
  Assess the AI components of a supply chain for the two properties that make
  them different — silent change, and running with agent authority — and
  invalidate the control tests that predate a model change. Use when a model
  provider changed the model and you need to know what that invalidates, and at
  vendor assessment or renewal.
allowed-tools: Read, Grep, Glob
---

# The vendor changed the model and your control tests expired

Third-party AI differs from ordinary third-party software in two ways. A hosted
model can change underneath you with no change record on your side, which
invalidates every control test taken before it. And a tool package or MCP
connector runs **with your agent's authority**, so its risk is not the
vendor's — it is yours.

## When to use this

Vendor assessment, renewal, and any time a provider announces a model update.

## Procedure

**1 — Enumerate the AI components.** Hosted models, tool packages, MCP servers,
embedded features in products you already bought. The last category is the one
nobody lists.

**2 — Score each on the two properties.** Can it change without telling you, and
does it execute with your agent's authority? Either one alone justifies a higher
tier than the ordinary assessment would give.

**3 — Record the last known model version, with a date.** Without it you cannot
tell whether a control test predates a change, which makes the next step
impossible.

**4 — Invalidate control tests taken before the change.** Not "review" — mark
them unevidenced. A test performed against a different model is not weak
evidence, it is evidence about something else.

**5 — Ask the questions a contract can answer.** Notice period for model change,
whether the version is pinnable, what telemetry you get, and exit. Then record
which ones the vendor declined; that list is the assessment.

## Output contract

```json
{
  "components": [{"name": "str", "kind": "model|tool|mcp|embedded",
                  "silent_change": true, "runs_with_agent_authority": false, "tier": "str"}],
  "versions": [{"component": "str", "version": "str", "as_of": "str", "changed_at": "str|null"}],
  "control_tests": [{"id": "str", "tested_at": "str", "status": "valid|unevidenced", "why": "str"}],
  "contract_questions": [{"question": "str", "answered": false}]
}
```

## Failure modes

- **Assessing the vendor and not the authority.** The connector runs as your
  agent.
- **Keeping a control test that predates a model change.** It evidences the old
  model.
- **Missing embedded AI features.** They arrived with a product you already
  own.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/grc/third-party-ai-assessment/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/grc/third-party-ai-assessment/scripts/third_party_ai_assessment.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Assess AI components of a supply chain for silent change and agent authority, and invalidate the control tests taken before a model changed.

This is the executable half of the `third-party-ai-assessment` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

from dataclasses import dataclass

@dataclass(frozen=True)
class Component:
    name: str; kind: str; signed: bool = False
    pinned: bool = False; can_change_silently: bool = False
    runs_with_agent_authority: bool = False; downloads: int = 0

COMPONENTS = [
 Component("cryptography==42.0.5", "library", True, True, False, False, 900_000),
 Component("langchain==0.2.1", "library", False, True, False, False, 400_000),
 Component("hosted GLM-4.6 endpoint", "hosted model", False, False, True, False),
 Component("local glm-4.6 weights (pinned digest)", "weights", True, True, False, False),
 Component("mcp-jira-connector==0.0.3", "tool package", False, True, False, True, 180),
]
def assess(c):
    flags = []
    if not c.signed:                  flags.append("unsigned")
    if not c.pinned:                  flags.append("not version-pinned")
    if c.can_change_silently:         flags.append("CAN CHANGE WITHOUT NOTICE")
    if c.runs_with_agent_authority:   flags.append("runs with agent authority")
    if c.kind == "library" and c.downloads < 1000: flags.append("little scrutiny")
    tier = ("high" if c.can_change_silently or c.runs_with_agent_authority
            else "medium" if flags else "low")
    return tier, flags

print(f"{'component':40s}{'kind':14s}{'tier':8s}flags")
print("-" * 96)
for c in COMPONENTS:
    tier, flags = assess(c)
    print(f"{c.name:40s}{c.kind:14s}{tier:8s}{', '.join(flags) or '—'}")

import time
now = time.time(); DAY = 86400

CONTROL_TESTS = {"SB-2": now - 20*DAY, "EV-2": now - 20*DAY, "DR-1": now - 20*DAY}
MODEL_CHANGED_AT = now - 5*DAY

print("your controls were tested against a model that changed 5 days ago:")
for cid, tested in CONTROL_TESTS.items():
    valid = tested > MODEL_CHANGED_AT
    print(f"   {cid}  tested {int((now-tested)/DAY)}d ago  "
          f"{'still valid' if valid else 'INVALIDATED by the model change'}")
invalidated = [c for c, t in CONTROL_TESTS.items() if t <= MODEL_CHANGED_AT]
print(f"\n{len(invalidated)}/{len(CONTROL_TESTS)} control tests invalidated by a "
      f"change you did not make and were not told about.")
assert invalidated

QUESTIONS = [
 ("Can this component change without notifying us?",
  "if yes, every control test has an implicit expiry tied to the vendor"),
 ("Does it execute with our agent's authority?",
  "if yes, assess it as code, not as a dependency"),
 ("Can we pin a digest, and do we?",
  "the difference between a supply chain and a subscription"),
 ("What is our exit if we stop using it?",
  "DORA Art.11 asks this directly; most AI contracts have no answer"),
]
for q, why in QUESTIONS: print(f"Q: {q}\n   → {why}\n")

SIGNALS = {
 "library":      {"signature": True, "downloads": True, "pinning": True, "lineage": True},
 "hosted model": {"signature": False, "downloads": False, "pinning": False, "lineage": False},
 "weights":      {"signature": True, "downloads": False, "pinning": True, "lineage": False},
 "tool package": {"signature": False, "downloads": False, "pinning": True, "lineage": False},
}
print(f"{'artefact class':16s}{'signals available':>20}  unavailable")
print("-" * 74)
for kind, sig in SIGNALS.items():
    have = [k for k, v in sig.items() if v]
    lack = [k for k, v in sig.items() if not v]
    print(f"{kind:16s}{f'{len(have)}/{len(sig)}':>20}  {lack or '—'}")

def assessment_statement(kind):
    sig = SIGNALS[kind]
    lack = [k for k, v in sig.items() if not v]
    return (f"{kind}: assessed on {len(sig)-len(lack)}/{len(sig)} signals. "
            f"{', '.join(lack) or 'none'} unavailable for this artefact class.")
print()
for kind in SIGNALS: print("  " + assessment_statement(kind))
print("\nThat last sentence is the deliverable. A rating that hides which signals")
print("were unavailable is a number someone will later rely on.")

## What you just proved

The hosted model and the MCP tool package both tier high — one for silent change, one for running with agent authority. The silent model change invalidates all three control tests taken before it. The signal table shows libraries with 4 of 4 signals available and hosted models with 0 of 4, and each assessment statement names what was unavailable.

## Your turn

Add "can this change without notifying us?" to your third-party assessment form. For hosted models the answer is usually yes, and it should carry an explicit control-test expiry.

---

**Next → [E1.9 · Model and agent lifecycle governance](https://spbreed.github.io/cyber-commons/lessons/E1.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*